In [1]:
# 전체 패키지 설치
!pip install -qU langchain langchain-openai langchain-community langchain-text-splitters

# 추가 유용한 패키지들
!pip install -q faiss-cpu tiktoken python-dotenv

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 120.4/120.4 kB 3.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 28.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 23.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.7/61.7 kB 2.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.1/73.1 kB 1.7 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.34.2 which is incompatible.
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.5/18.5 MB 80.0 MB/s eta 0:00:00


출력 파서 (Output Parser)
  - 모델의 출력을 처리하고, 그 결과를 원하는 형식으로 변환하는 역할을 한다
  - 모델에서 변환된 원시 텍스트를 분석하고, 특정정보를 출력하거나, 출력을 특정형식으로 재구성 한다
  - 주요기능
    1. 출력 포맷 변경
    2. 정보 추출
    3. 결과 정제
    4. 조건부 로직 적용
  - 종류
    1. csv Parser
    2. json Parser

In [ ]:
# CSV Parser
# -> CommaSeperatedListOutputParser

In [2]:
from langchain_core.output_parsers import CommaSeparatedListOutputParser

# CommaSeparatedListOutputParser 객체를 생성
output_parser = CommaSeparatedListOutputParser()


# 파서의 형식 지침을 가져온다. 이 지침은 모델에게 어떤 형식으로 응답해야 할지 알려준다.
format_instructions = output_parser.get_format_instructions()


# 형식 지침을 출력

print(format_instructions)

Your response should be a list of comma separated values, eg: `foo, bar, baz` or `foo,bar,baz`


In [3]:
import os
from google.colab import userdata

os.environ['OPENAI_API_KEY'] = userdata.get("OPENAI_API_KEY")

In [5]:
from langchain_core.prompts import PromptTemplate
from langchain_openai import ChatOpenAI
from langchain_core.output_parsers import CommaSeparatedListOutputParser

# 1. 파서 초기화
output_parser = CommaSeparatedListOutputParser()

# 2. 프롬프트 템플릿에 지침 포함
prompt = PromptTemplate.from_template(
    "다음 질문에 대해 5개의 항목을 쉼표로 구분하여 답변하세요.\n{format_instructions}\n질문: {question}"
)

# 3. 모델 정의
model = ChatOpenAI(model="gpt-4o-mini")

# 4. 체인 구성 (프롬프트 | 모델 | 파서)
chain = prompt | model | output_parser


# 5. 실행
result = chain.invoke({
    "question" : "한국의 대표적인 전통 음식 5가지를 알려줘",
    "format_instructions" : output_parser.get_format_instructions()
})

print(result)
# 출력 예시: ['비빔밥', '불고기', '김치', '잡채', '떡볶이']

['김치', '비빔밥', '불고기', '떡볶이', 'galbi']


- 실무 예시: JsonOutputParser 활용
단순히 텍스트를 받는 것보다, JSON 파서를 사용하여 AI가 구조화된 데이터를 뱉도록 강제하는 방식

In [6]:
from langchain_core.output_parsers import JsonOutputParser
from langchain_core.prompts import PromptTemplate
from langchain_openai import ChatOpenAI

# 1. 모델 설정
model = ChatOpenAI(model="gpt-4o-mini")

# 2. 파서 설정
parser = JsonOutputParser()


# 3. 프롬프트에 파서의 가이드라인(format instructions) 추가
prompt = PromptTemplate(
    template="질문에 대해 JSON 형식으로 답변하세요.\n{format_instructions}\n질문: {question}",
    input_variables=["question"],
    partial_variables={"format_instructions": parser.get_format_instructions()},
)

# 4. 체인 구성
chain = prompt | model | parser


# 5. 실행
result = chain.invoke({"question": "대한민국의 수도와 인구는?"})
print(result) # 이제 Python의 dict로 출력됩니다!

{'수도': '서울', '인구': 9800000}


In [7]:
from langchain_core.prompts import PromptTemplate
from langchain_openai import ChatOpenAI
from langchain_core.output_parsers import JsonOutputParser
from pydantic import BaseModel, Field

# 1. 한글 데이터 구조 정의 (Pydantic 모델)
class CusineRecipe(BaseModel):
    name: str = Field(description= "요리이름")
    recipe: str = Field(description= "요리하는 방법및 레시피")


# 2. JSON 파서 정의
output_parser = JsonOutputParser(pydantic_object= CusineRecipe)

format_instructions = output_parser.get_format_instructions()


# 3. 한글 프롬프트 템플릿 정의
prompt = PromptTemplate(
    template='사용자의 질문에 답변하세요.\n{format_instructions}\n질문: {query}\n',
    input_variables=['query'],
    partial_variables={'format_instructions': format_instructions}
)

# 4. 모델 초기화
model = ChatOpenAI(model='gpt-4o-mini', temperature=0 )

# 5. 체인 결합
chain = prompt | model | output_parser


# 6. 실행
response = chain.invoke({'query': '김치찌개 맛있게 끓이는 법 알려줘'})
print(response)

{'name': '김치찌개', 'recipe': '1. 재료 준비: 김치, 돼지고기, 두부, 대파, 마늘, 고춧가루, 국간장, 물을 준비합니다.\n2. 돼지고기를 먼저 볶습니다: 냄비에 돼지고기를 넣고 볶아 기름이 나올 때까지 익힙니다.\n3. 김치 추가: 볶은 돼지고기에 김치를 넣고 함께 볶아줍니다.\n4. 물 붓기: 볶은 재료에 물을 붓고 끓입니다.\n5. 양념하기: 끓기 시작하면 고춧가루와 국간장을 넣고 잘 섞습니다.\n6. 두부와 대파 추가: 두부를 넣고 대파와 마늘을 추가한 후, 중불에서 15-20분 정도 끓입니다.\n7. 맛 조절: 마지막으로 간을 보고 필요에 따라 소금을 추가합니다.\n8. 완성: 그릇에 담아 뜨겁게 서빙합니다.'}


# JSON 파서를 사용하여 AI가 구조화된 데이터를 뱉도록 강제하는 방식활용



문제) JsonOutputParser를 활용한 여행 일정 생성기
- 목표 :
사용자가 여행지와 여행 일수를 입력하면 AI가 여행 일정을 생성하고, JsonOutputParser를 사용하여 구조화된 JSON 형태로 반환하는 프로그램을 작성하세요.

In [ ]:
"""

요구사항
BaseModel을 이용하여 JSON 스키마를 정의하세요. -> Pydantic 모델
JsonOutputParser를 생성하세요.
PromptTemplate을 작성하세요.
parser.get_format_instructions()를 프롬프트에 포함하세요.
LCEL(|)을 사용하여 체인을 구성하세요.
invoke()를 이용하여 결과를 출력하세요.


입력 예시
여행지 : 제주도
여행일수 : 3


출력 예시
{
  "destination": "제주도",
  "days": 3,
  "schedule": [
    {
      "day": 1,
      "plan": "협재 해수욕장 방문"
    },
    {
      "day": 2,
      "plan": "성산일출봉 및 우도 관광"
    },
    {
      "day": 3,
      "plan": "동문시장 및 귀가"
    }
  ]
}

"""

In [12]:
# 라이브러리 불러오기

# ============================================================
# TODO 1
# 일정 정보를 저장할 BaseModel을 작성하세요.
# (day, plan)
# ============================================================
from pydantic import BaseModel, Field

class Schedule(BaseModel):
    day : int = Field(description="여행 날짜")
    plan : str = Field(description="해당 날짜의 여행 일정")

# ============================================================
# TODO 2
# 전체 여행 정보를 저장할 BaseModel을 작성하세요.
# (destination, days, schedule)
# ============================================================
from typing import List

class TravelPlan(BaseModel):
    destination : str = Field(description="여행지")
    days : int = Field(description="여행 일수")
    schedule : List[Schedule] = Field(description="여행 일정 목")

# ============================================================
# TODO 3
# JsonOutputParser를 생성하세요.
# ============================================================
from langchain_core.output_parsers import JsonOutputParser

parser = JsonOutputParser(pydantic_object= TravelPlan)

# ============================================================
# TODO 4
# PromptTemplate을 작성하세요.
#
# 반드시 포함할 변수
# - destination
# - days
#
# parser.get_format_instructions()를 포함하세요.
# ============================================================
from langchain_core.prompts import PromptTemplate

prompt = PromptTemplate(
    template= """
    당신은 국내 여행 전문가입니다.

    {format_instructions}

    여행지 : {destination}
    여행일수 : {days}

    조건
    - 하루에 일정은 2개만 작성
    - 현실적인 여행코스를 추천
    - 반드시 JSON만 출력
    """,
    input_variables= ["destination", "days"],
    partial_variables= {
        "format_instructions" : parser.get_format_instructions()
    },
)

# ============================================================
# TODO 5
# ChatOpenAI 객체를 생성하세요.
# ============================================================
from langchain_openai import ChatOpenAI

llm = ChatOpenAI(model = "gpt-4o-mini", temperature=0.7)

# ============================================================
# TODO 6
# LCEL 체인을 작성하세요.
#
# Prompt
# → ChatOpenAI
# → JsonOutputParser
# ============================================================

chain = prompt | llm | parser

# ============================================================
# TODO 7
# invoke()를 이용하여
# 제주도 / 3일 여행 일정을 생성하세요.
# ============================================================

result = chain.invoke({"destination": "제주도", "days" : 3,})

print("+"*60)
print("여행일정 출력")
print("+"*60)
print(result)

++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++
여행일정 출력
++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++
{'destination': '제주도', 'days': 3, 'schedule': [{'day': 1, 'plan': '제주 공항 도착 후 한라산 등반 시작, 저녁에는 제주 전통 시장 탐방'}, {'day': 1, 'plan': '제주 전통 시장에서 저녁식사 후, 숙소 체크인'}, {'day': 2, 'plan': '성산 일출봉에서 일출 감상, 근처 성산포 해변에서 산책'}, {'day': 2, 'plan': '우도 여행, 유채꽃밭과 해수욕장 방문'}, {'day': 3, 'plan': '만장굴 탐방 후, 제주 민속촌 방문'}, {'day': 3, 'plan': '제주도 유명 맛집에서 점심식사 후 제주 공항으로 이동'}]}


문제) JsonOutputParser를 활용한 AI 이력서 평가 시스템

- 목표 :
지원자의 이력서를 분석하여 AI가 구조화된 JSON으로 평가 결과를 반환하는 프로그램을 작성하세요.

In [ ]:
"""

요구사항
Pydantic 모델 정의
JsonOutputParser 사용
parser.get_format_instructions() 사용
LCEL 체인 사용
invoke() 사용
입력 예시
이름 : 홍길동

Python
LangChain
FastAPI

프로젝트 3개 수행
출력 예시
{
  "name": "홍길동",
  "career_level": "신입",
  "skills": [
    "Python",
    "LangChain",
    "FastAPI"
  ],
  "strengths": [
    "...",
    "..."
  ],
  "weaknesses": [
    "..."
  ],
  "score": 92,
  "interview_questions": [
    "...",
    "...",
    "..."
  ]
}

"""

In [15]:
from typing import List
from pydantic import BaseModel, Field

from langchain_core.output_parsers import JsonOutputParser
from langchain_core.prompts import PromptTemplate
from langchain_openai import ChatOpenAI


# ============================================================
# TODO 1
# ResumeEvaluation BaseModel을 작성하세요.
#
# 포함 항목
# name
# career_level
# skills
# strengths
# weaknesses
# score
# interview_questions
# ============================================================

class ResumeEvalution(BaseModel):
    name: str = Field(description="지원자 이름")
    career_level: str = Field(description="경력 수준")
    skills: List[str] = Field(description="보유 기술")
    strengths: List[str] = Field(description="강점")
    weaknesses: List[str] = Field(description="보완점")
    score: int = Field(description="종합 평가 점수(0~100)")
    interview_questions: List[str] = Field(description="예상 면접 질문")


# ============================================================
# TODO 2
# JsonOutputParser를 생성하세요.
# ============================================================

parser = JsonOutputParser(pydantic_object=ResumeEvalution)

# ============================================================
# TODO 3
# PromptTemplate을 작성하세요.
#
# parser.get_format_instructions()를 포함하세요.
# 입력 변수
# resume
# ============================================================

prompt = PromptTemplate(
    template = """
    당신은 IT기업의 채용 담당자 입니다.
    아래 이력서를 분석하여 평가하세요.

    {format_instructions}

    평가기준
    - 기술 점수 분석
    - 강점 분석
    - 부족한 부분 분석
    - 100점 만점으로 평가
    - 예상 면접 질문 3개 생성


    이력서
    --------------------
    {resume}
    --------------------

    반드시 JSON만 출력 하세요.
    """,
    input_variables=["resume"],
    partial_variables={
        "format_instructions" : parser.get_format_instructions()
    },
)

# ============================================================
# TODO 4
# ChatOpenAI 객체를 생성하세요.
# ============================================================

llm = ChatOpenAI(model = "gpt-4o-mini", temperature=0.5)

# ============================================================
# TODO 5
# LCEL 체인을 작성하세요.
# ============================================================

chain = prompt | llm | parser

# ============================================================
# TODO 6
# invoke()를 이용하여
# 예제 이력서를 분석하세요.
# ============================================================

resume = """
이름 : 홍길동

기술
- python
- langchain
- fastAPI
- SQL

프로젝트
- RAG 에이전트
- PDF QA 시스템 개발

경력
신입
"""

result = chain.invoke(
    {
        "resume" : resume
    }
)


print("="*60)
print("AI 이력서 평가 결과")
print("="*60)

print(f"이름 : {result['name']}")
print(f"경력 : {result['career_level']}")
print(f"점수 : {result['score']}")


print("\n[보유 기술]")
for skill in result["skills"]:
    print("-", skill)

print("\n[강점]")
for strengths in result["strengths"]:
    print("-", strengths)

print("\n[보완점]")
for weaknesses in result["weaknesses"]:
    print("-", weaknesses)

print("\n[예상 면접 질문]")
for i, question in enumerate(result["interview_questions"], start=1):
    print(f"{i}, {question}")

AI 이력서 평가 결과
이름 : 홍길동
경력 : 신입
점수 : 75

[보유 기술]
- python
- langchain
- fastAPI
- SQL

[강점]
- 신기술에 대한 빠른 학습 능력
- 문제 해결 능력
- 팀워크

[보완점]
- 실무 경험 부족
- 프로젝트 관리 경험 부족

[예상 면접 질문]
1, 당신이 가장 자신 있는 기술은 무엇이며, 그 이유는 무엇인가요?
2, 최근에 진행한 프로젝트에 대해 설명해 주세요.
3, 팀워크를 발휘한 경험에 대해 이야기해 주세요.


문제) JsonOutputParser를 활용한 고객 문의 자동 분류 시스템

- 목표 :
고객의 문의 내용을 AI가 분석하여 문의 유형, 우선순위, 감정 분석, 요약 및 추천 조치를 JSON 형태로 반환하는 프로그램을 작성하세요.

In [ ]:
# 전체 패키지 설치
!pip install -qU langchain langchain-openai langchain-community langchain-text-splitters

# 추가 유용한 패키지들
!pip install -q faiss-cpu tiktoken python-dotenv

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 120.4/120.4 kB 5.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 64.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 42.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.7/61.7 kB 3.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.1/73.1 kB 4.4 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.34.2 which is incompatible.
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.5/18.5 MB 62.7 MB/s eta 0:00:00


In [ ]:
"""

요구사항
BaseModel 정의
JsonOutputParser 사용
parser.get_format_instructions() 사용
PromptTemplate 작성
LCEL 체인 작성
invoke() 사용


입력 예시
주문한 상품이 아직 도착하지 않았습니다.
배송 조회도 되지 않습니다.


출력 예시
{
  "category": "배송",
  "priority": "높음",
  "sentiment": "부정",
  "summary": "배송 지연 및 배송 조회 불가 문의",
  "recommended_action": "배송사 운송장 상태를 확인하고 고객에게 배송 현황을 안내한다."
}

""

In [16]:
from typing import List
from pydantic import BaseModel, Field

from langchain_core.output_parsers import JsonOutputParser
from langchain_core.prompts import PromptTemplate
from langchain_openai import ChatOpenAI

# ============================================================
# TODO 1
# CustomerInquiry BaseModel을 작성하세요.
#
# 포함 항목
# category
# priority
# sentiment
# summary
# recommended_action
# ============================================================
class CustomerInquiry(BaseModel):
    category : str = Field(description="문의 유형")
    priority : str = Field(description="문의 우선 순위")
    sentiment : str = Field(description="고객 감정")
    summary : str = Field(description="문의 내용 요약")
    recommended_action : str = Field(description="담당자의 추천 조치")


# ============================================================
# TODO 2
# JsonOutputParser를 생성하세요.
# ============================================================

parser = JsonOutputParser(pydantic_object= CustomerInquiry)

# ============================================================
# TODO 3
# PromptTemplate을 작성하세요.
#
# parser.get_format_instructions()를 반드시 포함하세요.
#
# 입력 변수
# inquiry
# ============================================================
prompt = PromptTemplate(
    template = """
    당신은 온라인 쇼핑몰 고객센터 상담원입니다.

    아래 고객 문의를 분석하세요.

    {format_instructions}

    분석항목
    1. 문의 유형(category)
    2. 문의 우선 순위(priority)
    3. 고객 감정(sentiment)
    4. 문의 내용 요약(summary)
    5. 담당자의 추천 조치(recommended_action)

    고객문의
    ------------------
    {inquiry}
    ------------------

    반드시 JSON만 출력하세요.

    """,
    input_variables = ["inquiry"],
    partial_variables = {
        "format_instructions": parser.get_format_instructions()
    }
)


# ============================================================
# TODO 4
# ChatOpenAI 객체를 생성하세요.
# ============================================================

llm = ChatOpenAI(model= "gpt-4o", temperature=0)

# ============================================================
# TODO 5
# LCEL 체인을 작성하세요.
#
# Prompt
# → ChatOpenAI
# → JsonOutputParser
# ============================================================

chain = prompt | llm | parser

# ============================================================
# TODO 6
# invoke()를 이용하여
# 예제 고객 문의를 분석하세요.
# ============================================================

inquiry = """

주문한 노트북이 아직 도착하지 않았습니다.

배송 조회도되지 않고,
언제 받을 수 있는지도 확인이 되지 않습니다.

업체와 연락도 불가합니다.

빠른 처리 부탁드립니다.

"""

result = chain.invoke(
    {
        "inquiry" : inquiry
    }
)


print('-'*70)
print("고객의 문의 자동 분석 결과")
print('-'*70)


print(f"문의 유형   : {result['category']}")
print(f"우선 순위   : {result['priority']}")
print(f"고객 감정   : {result['sentiment']}")

print("\n[문의 요약]")
print(result["summary"])

print("\n[추천 조치]")
print(result["recommended_action"])

----------------------------------------------------------------------
고객의 문의 자동 분석 결과
----------------------------------------------------------------------
문의 유형   : 배송 문제
우선 순위   : 높음
고객 감정   : 불만

[문의 요약]
주문한 노트북이 아직 도착하지 않았고, 배송 조회 및 업체와의 연락이 불가함.

[추천 조치]
배송 상태 확인 후 고객에게 신속히 안내하고, 배송 지연 사과 및 해결 방안 제시
